# Fit Linear Regression Model with NumPy

* Data generation:

In [1]:
import numpy as np

In [6]:
n = 100
rv = np.random.RandomState(0)

x = rv.uniform(0,1,[n,1])
y = 1 + 2*x + 0.1*rv.normal(0,1,[n,1])


In [7]:
x, y

(array([[0.5488135 ],
        [0.71518937],
        [0.60276338],
        [0.54488318],
        [0.4236548 ],
        [0.64589411],
        [0.43758721],
        [0.891773  ],
        [0.96366276],
        [0.38344152],
        [0.79172504],
        [0.52889492],
        [0.56804456],
        [0.92559664],
        [0.07103606],
        [0.0871293 ],
        [0.0202184 ],
        [0.83261985],
        [0.77815675],
        [0.87001215],
        [0.97861834],
        [0.79915856],
        [0.46147936],
        [0.78052918],
        [0.11827443],
        [0.63992102],
        [0.14335329],
        [0.94466892],
        [0.52184832],
        [0.41466194],
        [0.26455561],
        [0.77423369],
        [0.45615033],
        [0.56843395],
        [0.0187898 ],
        [0.6176355 ],
        [0.61209572],
        [0.616934  ],
        [0.94374808],
        [0.6818203 ],
        [0.3595079 ],
        [0.43703195],
        [0.6976312 ],
        [0.06022547],
        [0.66676672],
        [0

In [8]:

idx = np.arange(n)
rv.shuffle(idx)
train_idx = idx[:int(0.8*n)]
test_idx = idx[int(0.8*n):]
x_train, y_train = x[train_idx], y[train_idx]
x_test, y_test = x[test_idx], y[test_idx]

* Fit linear model using numpy:

In [9]:
# initializes parameters "a" and "b" randomly
a = rv.normal(0,1,1)
b = rv.normal(0,1,1)

# define learning rate, number of epochs
lr = 1e-1
n_epochs = 1000

for epoch in range(n_epochs):
    # pathforward to calculate the loss
    yhat = a + b*x_train
    error = y_train - yhat
    loss = (error**2).mean()

    a_grad = -2*error.mean()
    b_grad = -2*(x_train*error).mean()

    a -= lr*a_grad
    b -= lr*b_grad

print(f'a={a}, b={b}')

a=[1.01785216], b=[2.00436531]


* Compare results with the `LinearRegression` module:

In [10]:
from sklearn.linear_model import LinearRegression
lm = LinearRegression()
lm.fit(x_train, y_train) # reshape the x into a 2D array
print(lm.intercept_,lm.coef_)

[1.01785108] [[2.00436756]]


* Changing the learning rate will change the results slightly.
* If the dataset is very large, we might need to split it into mini-batches.

# From NumPy to PyTorch

* Let's use PyTorch to reduce the work above.

In [2]:
import numpy as np
import torch


In [ ]:
torch.cuda.is_available() # 사용 가능한 GPU가 있는지 확인
#cuda : GPU에서 연산을 수행하도록 하는 장치 객체

## PyTorch Step 1: Use autograd to reduce the work

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
n_epochs = 1000 # 반복횟수
lr = 1e-1

In [11]:
# when create a new tensor, requires_grad often set as false, so we need to set to True explicitly.
a = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)
b = torch.randn(1, requires_grad=True, dtype=torch.float, device=device)

x_train_tensor = torch.from_numpy(x_train).to(device, dtype=torch.float)
y_train_tensor = torch.from_numpy(y_train).to(device, dtype=torch.float)

In [12]:
for epoch in range(n_epochs):
    yhat = a + b * x_train_tensor
    error = y_train_tensor - yhat
    loss = (error ** 2).mean()

    # No more manual computation of gradients!
    # a_grad = -2 * error.mean()
    # b_grad = -2 * (x_tensor * error).mean()
    # We just tell PyTorch to work its way BACKWARDS from the specified loss!
    loss.backward()

    with torch.no_grad():
        a -= lr * a.grad # <1>
        b -= lr * b.grad

    a.grad.zero_()
    b.grad.zero_()

print(a, b)

tensor([1.0178], requires_grad=True) tensor([2.0044], requires_grad=True)


* `a -= lr * a.grad` (i.e., <1>) is different than `a = a - lr*a.grad`, the `-=` replaces `a` in place, but the later one create a new `a`. When a new `a` is created, we lose its gradients.
* It is also important to set an enviorment with `torch.no_grad()` so that it will not add this update into computational graph.

## PyTorch Step 2: Use optimizer to reduce the work

In [13]:
import torch.optim as optim

In [14]:
optimizer = optim.SGD([a,b], lr=lr) #a,b 학습해야할 것, lr : learning rate 

for epoch in range(n_epochs):
    yhat = a + b * x_train_tensor
    error = y_train_tensor - yhat
    loss = (error ** 2).mean()

    # No more manual computation of gradients!
    # a_grad = -2 * error.mean()
    # b_grad = -2 * (x_tensor * error).mean()
    # We just tell PyTorch to work its way BACKWARDS from the specified loss!
    loss.backward()

    # with torch.no_grad():
    #     a -= lr * a.grad # <1>
    #     b -= lr * b.grad
    optimizer.step()
    # a.grad.zero_()
    # b.grad.zero_()
    optimizer.zero_grad()

print(a, b)

tensor([1.0178], requires_grad=True) tensor([2.0044], requires_grad=True)


## PyTorch Step 3: Use Loss to reduce the work

In [15]:
import torch.nn as nn

In [16]:
loss_fn = nn.MSELoss(reduction='mean')

for epoch in range(n_epochs):
    yhat = a + b * x_train_tensor

    # error = y_train_tensor - yhat
    # loss = (error ** 2).mean()
    loss = loss_fn(y_train_tensor, yhat)
    # No more manual computation of gradients!
    # a_grad = -2 * error.mean()
    # b_grad = -2 * (x_tensor * error).mean()
    # We just tell PyTorch to work its way BACKWARDS from the specified loss!
    loss.backward()

    # with torch.no_grad():
    #     a -= lr * a.grad # <1>
    #     b -= lr * b.grad
    optimizer.step()
    # a.grad.zero_()
    # b.grad.zero_()
    optimizer.zero_grad()

print(a, b)

tensor([1.0178], requires_grad=True) tensor([2.0044], requires_grad=True)


## PyTorch Step 4:Use model to reduce the work

In [17]:
class ManualLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Parameter(torch.randn(1, requires_grad=True, dtype=torch.float))
        self.b = nn.Parameter(torch.randn(1, requires_grad=True, dtype=torch.float))

    def forward(self, x):
        return self.a + self.b*x

In [18]:
# create the model and sent to device
model = ManualLinearRegression().to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
for epoch in range(n_epochs):
    # yhat = a + b * x_train_tensor
    model.train()
    yhat = model(x_train_tensor)
    # error = y_train_tensor - yhat
    # loss = (error ** 2).mean()
    loss = loss_fn(y_train_tensor, yhat)
    # No more manual computation of gradients!
    # a_grad = -2 * error.mean()
    # b_grad = -2 * (x_tensor * error).mean()
    # We just tell PyTorch to work its way BACKWARDS from the specified loss!
    loss.backward()

    # with torch.no_grad():
    #     a -= lr * a.grad # <1>
    #     b -= lr * b.grad
    optimizer.step()
    # a.grad.zero_()
    # b.grad.zero_()
    optimizer.zero_grad()

print(model.state_dict())

OrderedDict({'a': tensor([1.0179]), 'b': tensor([2.0044])})


## PyTorch Step 5: Improve writing the model using built-in models

In [19]:
class LayerLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1,1)  # Multiple 될 경우  linear (1,10) ?? 

    def forward(self, x):
        return self.linear(x)

In [20]:
model = LayerLinearRegression().to(device)
optimizer = optim.SGD(model.parameters(), lr=lr)
for epoch in range(n_epochs):
    model.train()
    yhat = model(x_train_tensor)
    loss = loss_fn(y_train_tensor, yhat)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print(model.state_dict())

OrderedDict({'linear.weight': tensor([[2.0044]]), 'linear.bias': tensor([1.0179])})


## PyTorch Step 6: Put model, loss_fn, and optimizer into a train step function

In [21]:
def make_train_step(model, loss_fn, optimizer):
    def train_step(x,y):
        model.train()
        yhat = model(x)
        loss = loss_fn(y, yhat)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        return loss.item()
    return train_step

In [22]:
train_step = make_train_step(model,loss_fn, optimizer)


In [23]:

losses =[]

for epoch in range(n_epochs):
    loss = train_step(x_train_tensor,y_train_tensor)
    losses.append(loss)
print(model.state_dict())

OrderedDict({'linear.weight': tensor([[2.0044]]), 'linear.bias': tensor([1.0179])})


## PyTorch Step 7: Use PyTorch utilities for effective data handling and partitioning

In [24]:
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

In [25]:
class CustomDataSet(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor

    def __getitem__(self, index): #index 
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)

In [26]:
x_tensor = torch.from_numpy(x).float()
y_tensor = torch.from_numpy(y).float()

dataset = CustomDataSet(x_tensor, y_tensor)
# same as dataset = TensorDataset(x_tensor, y_tensor)

# the above lines generate the same results,
# however when data are complicated or very large, we need to write our own dataset by loading data from disk.
#print(dataset[0])
#dataset = TensorDataset(x_tensor, y_tensor)
#print(dataset[0])

In [27]:
lengths = [int(len(dataset)*0.8), int(len(dataset)*0.2)]
train_dataset, test_dataset = random_split(dataset, lengths) # split as training and testing

In [28]:
train_loader = DataLoader(dataset = train_dataset, batch_size = 50, shuffle = True)
test_loader = DataLoader(dataset = test_dataset, batch_size = 50)

In [31]:
test_losses = [] 
for epoch in range(n_epochs):
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        loss = train_step(x_batch,y_batch)
        losses.append(loss)

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test = x_test.to(device)
            y_test = y_test.to(device)
            model.eval() #train 이후에 model을 eavlaution 관점에서 쓰려면 정리해줘야함 
            yhat = model(x_test)
            test_loss = loss_fn(y_test, yhat)
            test_losses.append(test_loss.item())

print(model.state_dict())

OrderedDict({'linear.weight': tensor([[2.0155]]), 'linear.bias': tensor([0.9965])})


In [32]:
test_losses

[0.008930239826440811,
 0.009142971597611904,
 0.00899670459330082,
 0.009198186919093132,
 0.009084776043891907,
 0.009601575322449207,
 0.009469450451433659,
 0.009694819338619709,
 0.009877675212919712,
 0.01010711956769228,
 0.009987326338887215,
 0.009729509241878986,
 0.009990688413381577,
 0.010192280635237694,
 0.009525744244456291,
 0.009534112177789211,
 0.009659901261329651,
 0.0094237569719553,
 0.009635594673454762,
 0.009644250385463238,
 0.009216061793267727,
 0.009321076795458794,
 0.009293654933571815,
 0.009323176927864552,
 0.009494069032371044,
 0.009722349233925343,
 0.0097587238997221,
 0.009482009336352348,
 0.009259807877242565,
 0.009040271863341331,
 0.009293007664382458,
 0.009394784457981586,
 0.009269447065889835,
 0.009617256931960583,
 0.009584939107298851,
 0.009499077685177326,
 0.009629731997847557,
 0.010011422447860241,
 0.009650452062487602,
 0.009842721745371819,
 0.0097346231341362,
 0.009792516008019447,
 0.00994378887116909,
 0.01017056219279766

# Final Version with PyTorch

This is the final PyTorch code for fitting a linear model. It demonstrates:

1. How to prepare the dataset and utilize a DataLoader
2. How to define make a train step function
3. How to integrate the model, loss function, and optimizer

In [33]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

In [34]:
"""
Generate Data
"""
n = 1000
rv = np.random.RandomState(0)
x = rv.uniform(0, 1, [n,1])
y = 1 + 2 * x + 0.1*rv.normal(0, 1, [n,1])

"""
Fit Model Using PyTorch
"""

device = 'cuda' if torch.cuda.is_available() else 'cpu'

"""
Prepare for the data
"""
class CustomDataSet(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor

    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)

x_tensor = torch.from_numpy(x).float()
y_tensor = torch.from_numpy(y).float()

# dataset = CustomDataSet(x_tensor, y_tensor)
dataset = TensorDataset(x_tensor, y_tensor)
lengths = [int(len(dataset)*0.8), int(len(dataset)*0.2)]
train_dataset, test_dataset = random_split(dataset, lengths)

train_loader = DataLoader(dataset = train_dataset, batch_size = 50, shuffle = True)
test_loader = DataLoader(dataset = test_dataset, batch_size = 50)

"""
Define Model, Loss Function, Optimizer
"""

class LayerLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1,1)

    def forward(self, x):
        return self.linear(x)

loss_fn = nn.MSELoss(reduction='mean')
model = LayerLinearRegression().to(device)
optimizer = optim.SGD(model.parameters(), lr = 1e-1)

"""
Make A Train Step
"""
def make_train_step(model, loss_fn, optimizer):
    def train_step(x, y):
        # set the model in train state
        model.train()
        yhat = model(x)
        loss = loss_fn(y, yhat)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        return loss.item()
    return train_step

""""Train The Model"""
losses = []
test_losses = []
n_epochs = 1000

train_step = make_train_step(model, loss_fn, optimizer)

for epoch in range(n_epochs):
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        loss = train_step(x_batch,y_batch)
        losses.append(loss)

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test = x_test.to(device)
            y_test = y_test.to(device)
            model.eval()
            yhat = model(x_test)
            test_loss = loss_fn(y_test, yhat)
            test_losses.append(test_loss.item())

print(model.state_dict())

OrderedDict({'linear.weight': tensor([[1.9831]]), 'linear.bias': tensor([1.0068])})
